In [ ]:
import random
import heapq
import time
from copy import deepcopy

def goal_state(n):
    return tuple(list(range(1, n*n)) + [0])

def print_state(state, n):
    for i in range(n):
        print(state[i*n:(i+1)*n])
    print()

def get_neighbors(state, n):
    idx = state.index(0)
    x, y = divmod(idx, n)
    moves = []
    directions = [(-1,0),(1,0),(0,-1),(0,1)]
    for dx, dy in directions:
        nx, ny = x + dx, y + dy
        if 0 <= nx < n and 0 <= ny < n:
            new_idx = nx*n + ny
            new_state = list(state)
            new_state[idx], new_state[new_idx] = new_state[new_idx], new_state[idx]
            moves.append(tuple(new_state))
    return moves

def hamming(state, goal):
    return sum(1 for i in range(len(state)) if state[i] != 0 and state[i] != goal[i])

def manhattan(state, n):
    dist = 0
    for i, val in enumerate(state):
        if val == 0:
            continue
        goal_x, goal_y = divmod(val-1, n)
        cur_x, cur_y = divmod(i, n)
        dist += abs(goal_x-cur_x) + abs(goal_y-cur_y)
    return dist

def random_state(n, moves=100):
    state = list(goal_state(n))
    for _ in range(moves):
        neigh = get_neighbors(tuple(state), n)
        state = list(random.choice(neigh))
    return tuple(state)

def reconstruct_path(came_from, current):
    path = []
    while current in came_from:
        path.append(current)
        current = came_from[current]
    path.append(current)
    return list(reversed(path))

def astar(start, n, heuristic):
    goal = goal_state(n)
    open_set = []
    heapq.heappush(open_set, (0, start))
    came_from = {}
    g_score = {start:0}
    visited = 0

    while open_set:
        _, current = heapq.heappop(open_set)
        visited += 1

        if current == goal:
            path = reconstruct_path(came_from, current)
            return path, visited

        for neighbor in get_neighbors(current, n):
            tentative = g_score[current] + 1
            if neighbor not in g_score or tentative < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative
                if heuristic == "manhattan":
                    f = tentative + manhattan(neighbor, n)
                else:
                    f = tentative + hamming(neighbor, goal)
                heapq.heappush(open_set, (f, neighbor))

    return None, visited

def ida_star(start, n, heuristic):
    goal = goal_state(n)
    visited = 0

    def h(state):
        if heuristic == "manhattan":
            return manhattan(state, n)
        return hamming(state, goal)

    def search(path, g, bound):
        nonlocal visited
        node = path[-1]
        f = g + h(node)
        if f > bound:
            return f
        if node == goal:
            return "FOUND"
        min_bound = float("inf")
        for neigh in get_neighbors(node, n):
            if neigh not in path:
                visited += 1
                path.append(neigh)
                t = search(path, g+1, bound)
                if t == "FOUND":
                    return "FOUND"
                if t < min_bound:
                    min_bound = t
                path.pop()
        return min_bound

    bound = h(start)
    path = [start]

    while True:
        t = search(path, 0, bound)
        if t == "FOUND":
            return path.copy(), visited
        if t == float("inf"):
            return None, visited
        bound = t

n = 3
algorithm = "A*"
heuristic = "manhattan"

start = random_state(n, moves=40)

print("Stan początkowy:")
print_state(start, n)

start_time = time.time()

if algorithm == "A*":
    solution, visited = astar(start, n, heuristic)
else:
    solution, visited = ida_star(start, n, heuristic)

end_time = time.time()

if solution:
    print("Długość rozwiązania:", len(solution)-1)
    print("Odwiedzone stany:", visited)
    print("Czas:", end_time - start_time)
    print("Lista kroków:")
    for step in solution:
        print_state(step, n)
else:
    print("Brak rozwiązania")


Stan początkowy:
(1, 2, 3)
(4, 0, 5)
(6, 7, 8)

Długość rozwiązania: 14
Odwiedzone stany: 86
Czas: 0.0007865428924560547
Lista kroków:
(1, 2, 3)
(4, 0, 5)
(6, 7, 8)

(1, 2, 3)
(4, 5, 0)
(6, 7, 8)

(1, 2, 3)
(4, 5, 8)
(6, 7, 0)

(1, 2, 3)
(4, 5, 8)
(6, 0, 7)

(1, 2, 3)
(4, 5, 8)
(0, 6, 7)

(1, 2, 3)
(0, 5, 8)
(4, 6, 7)

(1, 2, 3)
(5, 0, 8)
(4, 6, 7)

(1, 2, 3)
(5, 6, 8)
(4, 0, 7)

(1, 2, 3)
(5, 6, 8)
(4, 7, 0)

(1, 2, 3)
(5, 6, 0)
(4, 7, 8)

(1, 2, 3)
(5, 0, 6)
(4, 7, 8)

(1, 2, 3)
(0, 5, 6)
(4, 7, 8)

(1, 2, 3)
(4, 5, 6)
(0, 7, 8)

(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

